# Working with CLAMPS data — instrument overview example

This notebook walks through **every step** of building the gallery **4-panel instrument figure** for one case:

- **Case id:** `ci_c1` (NWCRIL2020 CLAMPS1, **2020-07-30 UTC**)
- **Gallery entry:** `gravity_waves_c1` (same observational day)

There are no project wrapper functions here — only standard Python, NumPy, Matplotlib, and `netCDF4` reads plus explicit array work and plotting.

**Bundled inputs** (after `scripts/link_data.sh`):

| Folder | Product | Used for |
|--------|---------|----------|
| `tropoe/` | TROPoe | θ_v and q (panels A & B); cloud-base markers |
| `dlfp/` | DLFP | Vertical-velocity variance σ²_w (panel D) |
| `dlvad/` | DLVAD | SNR ceiling on wind panel |
| `windoe/` | WINDoe | Horizontal wind (panel C) — precomputed |
| `pbl_fuzzy/` | fuzzy PBLH | Black PBL height line on all panels |

## Setup

In [ ]:
import os
import subprocess
import sys
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from matplotlib.colors import SymLogNorm
from netCDF4 import Dataset

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent):
    if (candidate / "scripts" / "link_data.sh").is_file():
        ROOT = candidate
        break

os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")
(ROOT / ".matplotlib").mkdir(exist_ok=True)

subprocess.run(["bash", str(ROOT / "scripts" / "link_data.sh")], check=True)
print(f"Project root: {ROOT}")

## 1. Case metadata and input file paths

We use one example case. File names include the UTC date (`20200730`). Each product lives in its own subfolder under `data/ci_c1/`.

In [ ]:
CASE_ID = "ci_c1"
CASE_DATE = datetime(2020, 7, 30, tzinfo=timezone.utc).date()
YMD = CASE_DATE.strftime("%Y%m%d")
PLATFORM = "C1"
CAMPAIGN = "NWCRIL 2020"
LOCATION = "Norman, OK"

DATA_ROOT = ROOT / "data" / CASE_ID
FIG_DIR = ROOT / "output" / "case_gallery" / "figures" / CASE_ID
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = FIG_DIR / "instrument_template_4panel.png"

def pick_case_file(folder: Path, ymd: str) -> Path:
    """Pick the case-day NetCDF file (archive inputs may use .cdf or .nc)."""
    if not folder.is_dir():
        raise FileNotFoundError(f"Missing data folder: {folder}")
    matches = sorted(
        p
        for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in {".nc", ".cdf"}
    )
    if not matches:
        raise FileNotFoundError(f"No NetCDF files (.nc/.cdf) in {folder}")
    dated = [p for p in matches if ymd in p.name]
    return (dated or matches)[-1]


paths = {}
for label, folder in [
    ("tropoe", DATA_ROOT / "tropoe"),
    ("dlfp", DATA_ROOT / "dlfp"),
    ("dlvad", DATA_ROOT / "dlvad"),
    ("windoe", DATA_ROOT / "windoe"),
    ("pbl", DATA_ROOT / "pbl_fuzzy"),
]:
    paths[label] = pick_case_file(folder, YMD)

tropoe_path = paths["tropoe"]
dlfp_path = paths["dlfp"]
dlvad_path = paths["dlvad"]
windoe_path = paths["windoe"]
pbl_path = paths["pbl"]

print(f"{CAMPAIGN} · {LOCATION} · CLAMPS{PLATFORM[-1]} · {CASE_DATE.isoformat()}")
print(f"TROPoe   {tropoe_path}")
print(f"DLFP     {dlfp_path}")
print(f"DLVAD    {dlvad_path}")
print(f"WINDoe   {windoe_path}")
print(f"PBL      {pbl_path}")

## 2. Read TROPoe and compute θ_v and q

TROPoe NetCDF variables used here:

- `hour` — decimal UTC hour (0–24)
- `height` — height (km)
- `temperature` — °C
- `waterVapor` — g kg⁻¹

Virtual potential temperature:

1. Convert temperature to Kelvin and mixing ratio to kg kg⁻¹.
2. Estimate pressure from height (standard atmosphere).
3. θ = T × (P₀/p)^κ, then θ_v = θ × (1 + 0.61 q).

In [ ]:
MAX_HEIGHT_KM = 2.0
P0_HPA = 1000.0
KAPPA = 0.286

with Dataset(tropoe_path) as nc:
    tropoe_hour = np.asarray(nc.variables["hour"][:], dtype=float)
    tropoe_height_km = np.asarray(nc.variables["height"][:], dtype=float)
    temperature_c = np.asarray(nc.variables["temperature"][:], dtype=float)
    water_vapor_gkg = np.asarray(nc.variables["waterVapor"][:], dtype=float)

h_mask = tropoe_height_km <= MAX_HEIGHT_KM
tropoe_height_km = tropoe_height_km[h_mask]
temperature_c = temperature_c[:, h_mask]
water_vapor_gkg = water_vapor_gkg[:, h_mask]

t_k = temperature_c + 273.15
q_kgkg = water_vapor_gkg / 1000.0
z_m = tropoe_height_km * 1000.0
p_hpa = 1013.25 * np.exp(-z_m / 8500.0)
theta = t_k * (P0_HPA / p_hpa[np.newaxis, :]) ** KAPPA
theta_v_k = theta * (1.0 + 0.61 * q_kgkg)

print(f"TROPoe: {tropoe_hour.size} profiles × {tropoe_height_km.size} heights (≤ {MAX_HEIGHT_KM:g} km)")
print(f"θ_v range (finite): {np.nanmin(theta_v_k):.1f} – {np.nanmax(theta_v_k):.1f} K")

## 3. Read DLFP and compute σ²_w on a regular time grid

DLFP provides vertical velocity (`velocity`) and return strength (`intensity`, SNR+1). We keep gates with `intensity ≥ 1.008`, then compute variance in 15-minute bins (0.25 h) with at least 10 samples per bin.

In [ ]:
DLFP_INTENSITY_MIN = 1.008
WVAR_BIN_HOURS = 0.25

with Dataset(dlfp_path) as nc:
    dlfp_hour = np.asarray(nc.variables["hour"][:], dtype=float)
    dlfp_height_km = np.asarray(nc.variables["height"][:], dtype=float)
    dlfp_velocity = np.asarray(nc.variables["velocity"][:], dtype=float)
    dlfp_intensity = np.asarray(nc.variables["intensity"][:], dtype=float)

h_mask = dlfp_height_km <= MAX_HEIGHT_KM
dlfp_height_km = dlfp_height_km[h_mask]
dlfp_velocity = dlfp_velocity[:, h_mask]
dlfp_intensity = dlfp_intensity[:, h_mask]

good = dlfp_intensity >= DLFP_INTENSITY_MIN
dlfp_velocity = np.where(good, dlfp_velocity, np.nan)

bin_edges = np.arange(0.0, 24.0 + WVAR_BIN_HOURS, WVAR_BIN_HOURS)
wvar_hour = bin_edges[:-1] + WVAR_BIN_HOURS / 2.0
w_variance = np.full((wvar_hour.size, dlfp_height_km.size), np.nan)
bin_index = np.digitize(dlfp_hour, bin_edges) - 1

for b in range(wvar_hour.size):
    in_bin = bin_index == b
    if np.sum(in_bin) < 10:
        continue
    w_variance[b, :] = np.nanvar(dlfp_velocity[in_bin, :], axis=0)

w_variance = np.where(w_variance > 0, w_variance, np.nan)
w_variance[w_variance > 5.0] = np.nan  # mask noise spikes

print(f"σ²_w grid: {wvar_hour.size} times × {dlfp_height_km.size} heights")

## 4. Read WINDoe horizontal wind

WINDoe is bundled precomputed for this example. Everywhere possible (this case included) the WINDoe algorithm ingests the PPI scan files, not the pre-computed VADs. WINDor Variables: `hour`, `height` (km), `u_wind`, `v_wind`, and speed uncertainty (`sigma_wspd`).

In [ ]:
with Dataset(windoe_path) as nc:
    wind_hour = np.asarray(nc.variables["hour"][:], dtype=float)
    wind_height_km = np.asarray(nc.variables["height"][:], dtype=float)
    u_wind = np.asarray(nc.variables["u_wind"][:], dtype=float)
    v_wind = np.asarray(nc.variables["v_wind"][:], dtype=float)
    if "sigma_wspd" in nc.variables:
        sigma_wspd = np.asarray(nc.variables["sigma_wspd"][:], dtype=float)
    else:
        sigma_wspd = np.asarray(nc.variables["sigma_w"][:], dtype=float)

wind_speed = np.hypot(u_wind, v_wind)

print(f"WINDoe: {wind_hour.size} times × {wind_height_km.size} heights")
print(f"Speed range (finite): {np.nanmin(wind_speed):.1f} – {np.nanmax(wind_speed):.1f} m s⁻¹")

## 5. Read fuzzy PBL height

As is the case for WINDoe, the fuzzy logic PBL code has already been run on these data and the output is staged for use here. The fuzzy-logic PBL product stores `t_epoch` (Unix seconds, UTC) and `pblh` (km). We convert to decimal UTC hour and meters for plotting.

In [ ]:
with Dataset(pbl_path) as nc:
    pbl_epoch = np.asarray(nc.variables["t_epoch"][:], dtype=float)
    pblh_km = np.asarray(nc.variables["pblh"][:], dtype=float)

pbl_hour = np.array([
    datetime.fromtimestamp(float(e), tz=timezone.utc).hour
    + datetime.fromtimestamp(float(e), tz=timezone.utc).minute / 60.0
    + datetime.fromtimestamp(float(e), tz=timezone.utc).second / 3600.0
    for e in pbl_epoch
], dtype=float)
pblh_m = pblh_km * 1000.0
pblh_m[pblh_m < 0] = np.nan

print(f"PBLH series: {pbl_hour.size} points")

## 6. DLVAD SNR ceiling and TROPoe cloud-base markers

**SNR ceiling:** for each DLVAD time, find the highest gate where `intensity ≥ 1.01`, bin to 15-minute UTC, then smooth with a 0.5 h rolling median for plotting.

While WINDoe uses the PPI files, not the VADs, we are using the VAD files to find the levels were traditional VAD-retreived wind profiles would not have enough signal to provide information. WINDoe is very useful for extracting signal out of the noise near this threshold. By plotting these levels, we can understand what WINDoe is 'buying' for us. 

**Cloud base:** TROPoe `cbh` (km) where `lwp > 5` g m⁻² (liquid water path gate).

The TROPoe retrieval is impacted by the presence of clouds. The retrieval will always assume there is a cloud somewhere in the atmosphere (unless the
user disables the cloud retrieval by setting both retrieve_lcloud and retrieve_icloud to zero in the
VIP file – this is NOT recommended). Thus, the algorithm needs to have some information on
how high to place this cloud, should it exist. With the codeployment of our IRS and Doppler lidar, we can use the Doppler lidar's fixed zeinth data to identify cloud bases and provide that to the algorithm. To avoid any confusion about the default cbh in the plots, we choose to only plot cbh when the LWP retreived from that profile is larger than 5 g/m. 

In [ ]:
DLVAD_SNR_THRESHOLD = 1.01
CLOUD_BASE_MIN_LWP = 5.0
CLOUD_BASE_MAX_KM = 1.8

with Dataset(dlvad_path) as nc:
    dlvad_hour = np.asarray(nc.variables["hour"][:], dtype=float)
    dlvad_height = np.asarray(nc.variables["height"][:], dtype=float)
    dlvad_intensity = np.asarray(nc.variables["intensity"][:], dtype=float)

dlvad_height_km = dlvad_height / 1000.0 if np.nanmax(dlvad_height) > 50.0 else dlvad_height

ceiling_km = np.full(dlvad_hour.size, np.nan)
for i in range(dlvad_hour.size):
    good = dlvad_intensity[i] >= DLVAD_SNR_THRESHOLD
    if np.any(good):
        ceiling_km[i] = float(np.max(dlvad_height_km[good]))

snr_df = pd.DataFrame({"hour": dlvad_hour, "ceiling": ceiling_km})
snr_df = snr_df[np.isfinite(snr_df["hour"]) & np.isfinite(snr_df["ceiling"])]
snr_df["tbin"] = (snr_df["hour"] / 0.25).round() * 0.25
snr_grouped = snr_df.groupby("tbin", as_index=False)["ceiling"].max()
snr_h = snr_grouped["tbin"].to_numpy()
snr_z = snr_grouped["ceiling"].to_numpy()
order = np.argsort(snr_h)
snr_h = snr_h[order]
snr_z = snr_z[order]

dt = float(np.median(np.diff(snr_h))) if snr_h.size > 1 else 0.25
win = max(1, int(round(0.5 / dt)))
snr_z = pd.Series(snr_z).rolling(win, center=True, min_periods=1).median().to_numpy()

with Dataset(tropoe_path) as nc:
    cb_hour = np.asarray(nc.variables["hour"][:], dtype=float)
    cbh = np.asarray(nc.variables["cbh"][:], dtype=float)
    lwp = np.asarray(nc.variables["lwp"][:], dtype=float) if "lwp" in nc.variables else None

cb_ok = np.isfinite(cb_hour) & np.isfinite(cbh) & (cbh > 0) & (cbh <= CLOUD_BASE_MAX_KM)
if lwp is not None:
    cb_ok &= np.isfinite(lwp) & (lwp > CLOUD_BASE_MIN_LWP)
cb_h = cb_hour[cb_ok]
cb_z = cbh[cb_ok]
cb_order = np.argsort(cb_h)
cb_h = cb_h[cb_order]
cb_z = cb_z[cb_order]

print(f"SNR ceiling: {snr_h.size} points")
print(f"Cloud-base markers (lwp > {CLOUD_BASE_MIN_LWP:g} g m⁻²): {cb_h.size}")

## 7. Plot settings

Color limits for this case match the gallery table. The x-axis spans 00–24 UTC on the case date.

In [ ]:
THETA_V_LIM = (297, 315)
Q_LIM = (5, 28)
WSPD_LIM = (3, 18)
SIGMA_WSPD_MAX = 3.0

period_start = datetime(CASE_DATE.year, CASE_DATE.month, CASE_DATE.day, tzinfo=timezone.utc)
period_end = period_start + timedelta(hours=24)
period_hours = 24.0

tropoe_name = tropoe_path.name.lower()
if "aeri_mwr" in tropoe_name:
    trop_inst = "AERI+MWR"
elif ".mwr." in tropoe_name:
    trop_inst = "MWR"
elif ".aeri." in tropoe_name or "aerioe" in tropoe_name:
    trop_inst = "AERI"
else:
    trop_inst = None

theta_title = rf"$\theta_v$ (TROPoe / {trop_inst})" if trop_inst else r"$\theta_v$ (TROPoe)"
q_title = rf"$q$ (TROPoe / {trop_inst})" if trop_inst else r"$q$ (TROPoe)"
wind_title = "Horizontal wind (WINDoe / DL-PPI)"
suptitle = (
    f"{CAMPAIGN} · {LOCATION} · CLAMPS{PLATFORM[-1]}\n"
    f"{CASE_DATE.isoformat()} · CLAMPS observations · PBLH (fuzzy logic)"
)

print(theta_title)
print(q_title)
print(wind_title)
print(f"Output: {OUT_PATH}")

## 8. Build the four-panel figure

Each panel uses `pcolormesh` on cell edges derived from profile center coordinates. Overlays: fuzzy PBLH (black line, white outline), cloud-base dots (panels A & B), DLVAD SNR ceiling (panel C), and wind barbs colored by σ_|V|.

In [ ]:
# --- helpers for pcolormesh edges (center coordinates → cell boundaries) ---
centers = np.asarray(tropoe_height_km, dtype=float)
centers = centers[np.isfinite(centers) & (centers <= MAX_HEIGHT_KM + 1e-9)]
if centers.size == 0:
    trop_z_edges = np.array([0.0, MAX_HEIGHT_KM])
elif centers.size == 1:
    trop_z_edges = np.array([centers[0] - 0.05, centers[0] + 0.05])
else:
    trop_z_edges = np.empty(centers.size + 1)
    trop_z_edges[1:-1] = 0.5 * (centers[:-1] + centers[1:])
    trop_z_edges[0] = centers[0] - (trop_z_edges[1] - centers[0])
    trop_z_edges[-1] = centers[-1] + (centers[-1] - trop_z_edges[-2])

if tropoe_hour.size == 0:
    trop_t_edges = np.array([0.0, 24.0])
elif tropoe_hour.size == 1:
    trop_t_edges = np.array([tropoe_hour[0] - 0.05, tropoe_hour[0] + 0.05])
else:
    trop_t_edges = np.empty(tropoe_hour.size + 1)
    trop_t_edges[1:-1] = 0.5 * (tropoe_hour[:-1] + tropoe_hour[1:])
    trop_t_edges[0] = tropoe_hour[0] - (trop_t_edges[1] - tropoe_hour[0])
    trop_t_edges[-1] = tropoe_hour[-1] + (tropoe_hour[-1] - trop_t_edges[-2])

fig, axes = plt.subplots(4, 1, figsize=(12, 13), sharex=True, layout="constrained")

# --- Panel A: θ_v ---
ax = axes[0]
cmap = plt.get_cmap("RdYlBu_r").copy()
cmap.set_under(cmap(0.0))
cmap.set_over(cmap(1.0))
pcm_a = ax.pcolormesh(
    trop_t_edges, trop_z_edges, theta_v_k.T,
    cmap=cmap, vmin=THETA_V_LIM[0], vmax=THETA_V_LIM[1], shading="flat",
)
pbl_ok = np.isfinite(pbl_hour) & np.isfinite(pblh_m)
if np.sum(pbl_ok) >= 2:
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="white", lw=3.5, zorder=9)
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="black", lw=2.0, zorder=10)
if cb_h.size:
    ax.scatter(cb_h, cb_z, s=28, facecolors="black", edgecolors="white", linewidths=0.9, zorder=12)
ax.set_ylim(0, MAX_HEIGHT_KM)
ax.set_ylabel("Height (km)")
ax.set_title(theta_title, fontsize=9)
fig.colorbar(pcm_a, ax=ax, label=r"$\theta_v$ (K)", fraction=0.03, pad=0.025, extend="both")
ax.text(0.02, 0.98, "A", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="none"), zorder=20)

# --- Panel B: q ---
ax = axes[1]
cmap = plt.get_cmap("YlGnBu").copy()
cmap.set_under(cmap(0.0))
cmap.set_over(cmap(1.0))
q_gkg = q_kgkg * 1000.0
pcm_b = ax.pcolormesh(
    trop_t_edges, trop_z_edges, q_gkg.T,
    cmap=cmap, vmin=Q_LIM[0], vmax=Q_LIM[1], shading="flat",
)
if np.sum(pbl_ok) >= 2:
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="white", lw=3.5, zorder=9)
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="black", lw=2.0, zorder=10)
if cb_h.size:
    ax.scatter(cb_h, cb_z, s=28, facecolors="black", edgecolors="white", linewidths=0.9, zorder=12)
ax.set_ylim(0, MAX_HEIGHT_KM)
ax.set_ylabel("Height (km)")
ax.set_title(q_title, fontsize=9)
fig.colorbar(pcm_b, ax=ax, label=r"$q$ (g kg$^{-1}$)", fraction=0.03, pad=0.025, extend="both")
ax.text(0.02, 0.98, "B", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="none"), zorder=20)

# --- Panel C: WINDoe wind speed + barbs ---
ax = axes[2]
h_mask = wind_height_km <= MAX_HEIGHT_KM
height_w = wind_height_km[h_mask]
wspd = wind_speed[:, h_mask]
u = u_wind[:, h_mask]
v = v_wind[:, h_mask]
sigma = sigma_wspd[:, h_mask]

if wind_hour.size == 1:
    wind_t_edges = np.array([wind_hour[0] - 0.05, wind_hour[0] + 0.05])
else:
    wind_t_edges = np.empty(wind_hour.size + 1)
    wind_t_edges[1:-1] = 0.5 * (wind_hour[:-1] + wind_hour[1:])
    wind_t_edges[0] = wind_hour[0] - (wind_t_edges[1] - wind_hour[0])
    wind_t_edges[-1] = wind_hour[-1] + (wind_hour[-1] - wind_t_edges[-2])

if height_w.size == 1:
    wind_z_edges = np.array([height_w[0] - 0.05, height_w[0] + 0.05])
else:
    wind_z_edges = np.empty(height_w.size + 1)
    wind_z_edges[1:-1] = 0.5 * (height_w[:-1] + height_w[1:])
    wind_z_edges[0] = height_w[0] - (wind_z_edges[1] - height_w[0])
    wind_z_edges[-1] = height_w[-1] + (height_w[-1] - wind_z_edges[-2])

valid = np.isfinite(u) & np.isfinite(v) & np.isfinite(wspd)
wspd_plot = np.where(valid, wspd, np.nan).copy()
for i in range(len(wind_hour)):
    gap_before = i > 0 and (wind_hour[i] - wind_hour[i - 1]) > 0.75
    gap_after = i < len(wind_hour) - 1 and (wind_hour[i + 1] - wind_hour[i]) > 0.75
    if gap_before or gap_after:
        wspd_plot[i, :] = np.nan

cmap = plt.get_cmap("viridis").copy()
cmap.set_under(cmap(0.0))
cmap.set_over(cmap(1.0))
pcm_c = ax.pcolormesh(
    wind_t_edges, wind_z_edges, wspd_plot.T,
    cmap=cmap, vmin=WSPD_LIM[0], vmax=WSPD_LIM[1], shading="flat",
)

high_sigma = np.isfinite(sigma) & (sigma > SIGMA_WSPD_MAX) & valid
if np.any(high_sigma):
    overlay = np.where(high_sigma, wspd_plot, np.nan)
    ax.pcolormesh(
        wind_t_edges, wind_z_edges, overlay.T,
        cmap="viridis", vmin=WSPD_LIM[0], vmax=WSPD_LIM[1], shading="flat", alpha=0.7,
        zorder=pcm_c.get_zorder() + 1,
    )

step_t = max(1, wspd.shape[0] // 20)
step_z = max(1, wspd.shape[1] // 10)
barb_high_sigma = np.isfinite(sigma) & (sigma > 2.5)
for i in range(0, wspd.shape[0], step_t):
    for j in range(0, wspd.shape[1], step_z):
        if not (valid[i, j]):
            continue
        if barb_high_sigma[i, j]:
            barbcolor = mcolors.to_rgba("black", 0.65)
            edgecolor = mcolors.to_rgba("white", 0.65)
        else:
            barbcolor = mcolors.to_rgba("white", 0.8)
            edgecolor = mcolors.to_rgba("black", 0.8)
        barbs = ax.barbs(
            [wind_hour[i]], [height_w[j]], [u[i, j]], [v[i, j]],
            length=5.5, barbcolor=barbcolor, flagcolor=barbcolor, linewidth=1.0, zorder=13,
        )
        barbs.set_path_effects([pe.Stroke(linewidth=2.8, foreground=edgecolor), pe.Normal()])

if snr_h.size >= 2:
    snr_order = np.argsort(snr_h)
    ax.plot(snr_h[snr_order], snr_z[snr_order], color="0.55", ls="--", lw=1.4, zorder=11)
if np.sum(pbl_ok) >= 2:
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="white", lw=3.5, zorder=17)
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="black", lw=2.0, zorder=18)
ax.set_ylim(0, MAX_HEIGHT_KM)
ax.set_ylabel("Height (km)")
ax.set_title(wind_title, fontsize=9)
fig.colorbar(pcm_c, ax=ax, label="Wind speed (m s$^{-1}$)", fraction=0.03, pad=0.025, extend="both")
ax.text(0.02, 0.98, "C", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="none"), zorder=20)

# --- Panel D: σ²_w ---
ax = axes[3]
wvar_z = dlfp_height_km.copy()
wvar_z = wvar_z[np.isfinite(wvar_z) & (wvar_z <= MAX_HEIGHT_KM + 1e-9)]
if wvar_z.size == 1:
    wvar_z_edges = np.array([wvar_z[0] - 0.05, wvar_z[0] + 0.05])
else:
    wvar_z_edges = np.empty(wvar_z.size + 1)
    wvar_z_edges[1:-1] = 0.5 * (wvar_z[:-1] + wvar_z[1:])
    wvar_z_edges[0] = wvar_z[0] - (wvar_z_edges[1] - wvar_z[0])
    wvar_z_edges[-1] = wvar_z[-1] + (wvar_z[-1] - wvar_z_edges[-2])

if wvar_hour.size == 1:
    wvar_t_edges = np.array([wvar_hour[0] - 0.05, wvar_hour[0] + 0.05])
else:
    wvar_t_edges = np.empty(wvar_hour.size + 1)
    wvar_t_edges[1:-1] = 0.5 * (wvar_hour[:-1] + wvar_hour[1:])
    wvar_t_edges[0] = wvar_hour[0] - (wvar_t_edges[1] - wvar_hour[0])
    wvar_t_edges[-1] = wvar_hour[-1] + (wvar_hour[-1] - wvar_t_edges[-2])

wvar_norm = SymLogNorm(linthresh=0.1, vmin=1e-3, vmax=10.0, base=10, linscale=1.0)
pcm_d = ax.pcolormesh(
    wvar_t_edges, wvar_z_edges, w_variance.T,
    cmap="summer", norm=wvar_norm, shading="flat",
)
if np.sum(pbl_ok) >= 2:
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="white", lw=3.5, zorder=9)
    ax.plot(pbl_hour[pbl_ok], pblh_m[pbl_ok] / 1000.0, color="black", lw=2.0, zorder=10)
ax.set_ylim(0, MAX_HEIGHT_KM)
ax.set_ylabel("Height (km)")
ax.set_title(r"$\sigma_w^2$ (DL-FP)", fontsize=9)
cbar_d = fig.colorbar(
    pcm_d, ax=ax, label=r"$\sigma_w^2$ (m$^2$ s$^{-2}$)", fraction=0.03, pad=0.025, extend="both",
)
cbar_d.set_ticks([1e-2, 1e-1, 1.0, 10.0])
cbar_d.set_ticklabels([r"$10^{-2}$", r"$10^{-1}$", r"$10^{0}$", r"$10^{1}$"])
ax.text(0.02, 0.98, "D", transform=ax.transAxes, fontsize=12, fontweight="bold", va="top",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="none"), zorder=20)

# --- Shared UTC x-axis (00–24 h) ---
tick_hours = np.arange(0, period_hours + 0.1, 3)
tick_labels = [
    (period_start + timedelta(hours=float(h))).strftime("%m-%d\n%HZ")
    for h in tick_hours
]
for ax in axes:
    ax.set_xlim(0, period_hours)
    ax.set_xticks(tick_hours)
    ax.axvspan(0.0, 24.0, color="0.92", alpha=0.22, zorder=0)
axes[-1].set_xticklabels(tick_labels, fontsize=8)
axes[-1].set_xlabel("UTC time", labelpad=10)
for ax in axes[:-1]:
    ax.set_xticklabels([])
    ax.tick_params(axis="x", labelbottom=False)

fig.suptitle(suptitle, fontsize=10)
fig.savefig(OUT_PATH, dpi=150, bbox_inches="tight", pad_inches=0.2)
plt.close(fig)
print(f"Saved {OUT_PATH}")

In [ ]:
display(Image(filename=str(OUT_PATH)))

## Next steps

- To use this notebook for your own data, you would need to point `DATA_ROOT` at another case's staged dataset (only one case is provided in repo) and update `CASE_ID`, `CASE_DATE`, and color limits.
- The more pythonic batch plotting used for all gallery cases shown on the gallery lives in `code/case_gallery/plot_instrument.py` (wrapper around the same steps).
- To see how the full pipeline works (THREDDS download, running WINDoe and fuzzy PBL algorithms): see the `clamps_viz_process` processing workspace.